# Case Técnico — Seção 1 | EDA e preparação para previsão diária

**Contexto:** histórico de vendas do E-commerce entre 01/11/2025 e 30/06/2026, agregado por **hora × categoria × canal**.

Objetivos da EDA, conforme o case:
- avaliar qualidade e consistência dos dados;
- estudar **preço médio de venda**;
- medir o **potencial de vendas por dia da semana**;
- entender **flutuações de comportamento ao longo do dia**;
- extrair insights de negócio;
- preparar uma visão diária consistente para a etapa de Machine Learning.

### Definição analítica adotada
Como o enunciado usa o termo **demanda diária**, a variável `qt_material` é tratada como a medida operacional mais direta de demanda. `receita_aprovada` permanece como KPI de negócio central e será analisada em paralelo. Na etapa de modelagem, a escolha do target será explicitamente justificada e poderá ser comparada com receita como análise de sensibilidade.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gb_ml.case_espec_i import (
    aggregate_daily,
    hourly_summary,
    load_case_csv,
    prepare_case_data,
    quality_summary,
    valid_categories,
    weekday_summary,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "Dados - Case Técnico Espec I.csv"

## 1. Leitura e entendimento do dataset

In [ ]:
df_raw = load_case_csv(DATA_PATH)
df = prepare_case_data(df_raw)

print(f"Linhas: {len(df):,}")
print(f"Colunas originais: {df_raw.shape[1]}")
print(f"Período: {df['dt_hr_venda'].min()} até {df['dt_hr_venda'].max()}")
display(df_raw.head())

### 1.1 Dicionário dos campos entregues

| Campo | Papel na análise |
|---|---|
| `dt_hr_venda` | timestamp da venda agregada |
| `DES_CANAL_VENDA_FINAL_AGRUP` | origem da compra: Site ou App |
| `DES_CATEGORIA_MATERIAL` | categoria do produto |
| `receita_aprovada` | receita aprovada agregada |
| `nr_pedidos` | quantidade de pedidos |
| `qt_material` | itens vendidos / demanda em unidades |
| `vlr_venda_desconto` | valor de desconto agregado |

## 2. Data Quality — antes de qualquer insight ou modelo

In [ ]:
dq = quality_summary(df)
display(dq)

### 2.1 Tipos, nulos e grão

O grão esperado pela descrição do arquivo é **hora × canal × categoria**. Primeiro confirmamos que o mesmo registro não aparece duplicado e que os tipos são coerentes.

In [ ]:
grain = [
    "dt_hr_venda",
    "DES_CANAL_VENDA_FINAL_AGRUP",
    "DES_CATEGORIA_MATERIAL",
]

schema_report = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "nulos": df_raw.isna().sum(),
    "pct_nulos": df_raw.isna().mean(),
    "unicos": df_raw.nunique(dropna=False),
})
display(schema_report)

print("Duplicidades no grão:", int(df.duplicated(grain).sum()))

### 2.2 Problema real encontrado: categoria corrompida

A coluna de categoria deveria conter nomes de categorias. No arquivo há valores numéricos entre 0 e 1 armazenados como texto. Esses registros são marcados como inválidos.

**Decisão:** não imputar categoria sem evidência suficiente. As linhas continuam nos totais de venda/demanda, mas são excluídas apenas de análises que exigem identificação correta da categoria.

In [ ]:
categoria_quality = pd.DataFrame({
    "metrica": [
        "linhas totais",
        "categorias distintas brutas",
        "categorias válidas distintas",
        "linhas com categoria inválida",
        "% linhas com categoria inválida",
    ],
    "valor": [
        len(df),
        df["DES_CATEGORIA_MATERIAL"].nunique(),
        len(valid_categories(df)),
        int(df["categoria_invalida"].sum()),
        float(df["categoria_invalida"].mean()),
    ],
})
display(categoria_quality)

print("Categorias válidas identificadas:")
print(valid_categories(df))

In [ ]:
categoria_invalida_mes = (
    df.groupby("mes", as_index=False)
    .agg(
        linhas=("categoria_invalida", "size"),
        categorias_invalidas=("categoria_invalida", "sum"),
        pct_invalida=("categoria_invalida", "mean"),
    )
)
display(categoria_invalida_mes)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(categoria_invalida_mes["mes"], categoria_invalida_mes["pct_invalida"] * 100, marker="o")
ax.set_title("Categoria inválida ao longo do tempo")
ax.set_ylabel("% de linhas")
ax.set_xlabel("Mês")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

### 2.3 Receita negativa

`receita_aprovada` contém valores negativos. Sem uma regra de negócio que confirme erro, esses registros **não serão removidos automaticamente**: podem representar estornos, devoluções ou ajustes.

Para métricas monetárias, mostramos o impacto; para o target operacional de demanda (`qt_material`), os itens continuam válidos.

In [ ]:
negative_summary = pd.DataFrame({
    "metrica": [
        "linhas com receita negativa",
        "% linhas com receita negativa",
        "soma receita negativa",
        "soma receita positiva",
        "impacto abs. negativo / positivo",
        "linhas com receita zero",
    ],
    "valor": [
        int(df["receita_negativa"].sum()),
        float(df["receita_negativa"].mean()),
        df.loc[df["receita_negativa"], "receita_aprovada"].sum(),
        df.loc[~df["receita_negativa"], "receita_aprovada"].sum(),
        -df.loc[df["receita_negativa"], "receita_aprovada"].sum()
        / df.loc[~df["receita_negativa"], "receita_aprovada"].sum(),
        int(df["receita_zero"].sum()),
    ],
})
display(negative_summary)

### 2.4 Cobertura temporal

O período deve conter 242 dias completos. Também verificamos continuidade horária e cobertura por canal.

In [ ]:
all_hours = pd.date_range(
    df["dt_hr_venda"].min(),
    df["dt_hr_venda"].max(),
    freq="h",
)
present_hours = pd.DatetimeIndex(df["dt_hr_venda"].unique())
missing_hours = all_hours.difference(present_hours)

hour_channel = (
    df[["dt_hr_venda", "DES_CANAL_VENDA_FINAL_AGRUP"]]
    .drop_duplicates()
    .set_index(["dt_hr_venda", "DES_CANAL_VENDA_FINAL_AGRUP"])
)
expected_hour_channel = pd.MultiIndex.from_product(
    [all_hours, ["App", "Site"]],
    names=["dt_hr_venda", "DES_CANAL_VENDA_FINAL_AGRUP"],
)
missing_hour_channel = expected_hour_channel.difference(hour_channel.index)

coverage = pd.DataFrame({
    "metrica": [
        "dias observados",
        "horas esperadas",
        "horas sem qualquer registro",
        "combinações hora-canal ausentes",
    ],
    "valor": [
        df["data"].nunique(),
        len(all_hours),
        len(missing_hours),
        len(missing_hour_channel),
    ],
})
display(coverage)

if len(missing_hour_channel):
    display(pd.DataFrame(missing_hour_channel.tolist(), columns=["dt_hr_venda", "canal"]))

## 3. Construção da visão diária

In [ ]:
daily = aggregate_daily(df)

print(f"Quantidade de dias: {len(daily)}")
display(daily.head())
display(
    daily[
        ["receita_aprovada", "nr_pedidos", "qt_material", "preco_medio_item", "ticket_medio"]
    ].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).T
)

### 3.1 Evolução de demanda e receita

Eventos extremos são mantidos: em e-commerce, picos podem representar campanhas reais. O objetivo é identificá-los e entendê-los, não removê-los por regra estatística.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily["data"], daily["qt_material"], label="Itens vendidos")
ax.plot(
    daily["data"],
    daily["qt_material"].rolling(7, min_periods=1).mean(),
    label="Média móvel 7 dias",
)
ax.plot(
    daily["data"],
    daily["qt_material"].rolling(28, min_periods=1).mean(),
    label="Média móvel 28 dias",
)
ax.set_title("Demanda diária — qt_material")
ax.set_xlabel("Data")
ax.set_ylabel("Itens vendidos")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily["data"], daily["receita_aprovada"], label="Receita aprovada")
ax.plot(
    daily["data"],
    daily["receita_aprovada"].rolling(7, min_periods=1).mean(),
    label="Média móvel 7 dias",
)
ax.set_title("Receita aprovada diária")
ax.set_xlabel("Data")
ax.set_ylabel("Receita aprovada")
ax.legend()
fig.tight_layout()
plt.show()

### 3.2 Dias extremos

Boxplots servem para sinalizar dias fora do padrão. Eles não definem exclusão automática.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(daily["qt_material"], vert=False)
axes[0].set_title("Boxplot — demanda diária")
axes[0].set_xlabel("qt_material")

axes[1].boxplot(daily["receita_aprovada"], vert=False)
axes[1].set_title("Boxplot — receita diária")
axes[1].set_xlabel("receita_aprovada")

fig.tight_layout()
plt.show()

print("Top 10 dias por demanda:")
display(
    daily.nlargest(10, "qt_material")[
        ["data", "dia_semana", "qt_material", "nr_pedidos", "receita_aprovada", "preco_medio_item"]
    ]
)

## 4. Insights solicitados no enunciado

### 4.1 Preço médio de venda

O preço médio é calculado de forma **ponderada**: `soma(receita_aprovada) / soma(qt_material)`. Isso evita o viés de tirar a média simples de preços médios de linhas com volumes muito diferentes.

In [ ]:
kpis = pd.DataFrame({
    "KPI": [
        "Receita aprovada total",
        "Pedidos",
        "Itens vendidos",
        "Preço médio por item",
        "Ticket médio",
        "Itens por pedido",
    ],
    "Valor": [
        df["receita_aprovada"].sum(),
        df["nr_pedidos"].sum(),
        df["qt_material"].sum(),
        df["receita_aprovada"].sum() / df["qt_material"].sum(),
        df["receita_aprovada"].sum() / df["nr_pedidos"].sum(),
        df["qt_material"].sum() / df["nr_pedidos"].sum(),
    ],
})
display(kpis)

In [ ]:
channel_summary = (
    df.groupby("DES_CANAL_VENDA_FINAL_AGRUP", as_index=False)
    .agg(
        receita=("receita_aprovada", "sum"),
        pedidos=("nr_pedidos", "sum"),
        itens=("qt_material", "sum"),
        desconto=("vlr_venda_desconto", "sum"),
    )
)
channel_summary["share_receita"] = channel_summary["receita"] / channel_summary["receita"].sum()
channel_summary["preco_medio_item"] = channel_summary["receita"] / channel_summary["itens"]
channel_summary["ticket_medio"] = channel_summary["receita"] / channel_summary["pedidos"]
display(channel_summary)

In [ ]:
valid_category_df = df.loc[~df["categoria_invalida"]].copy()

category_summary = (
    valid_category_df.groupby("categoria_material_limpa", as_index=False)
    .agg(
        receita=("receita_aprovada", "sum"),
        pedidos=("nr_pedidos", "sum"),
        itens=("qt_material", "sum"),
        desconto=("vlr_venda_desconto", "sum"),
    )
)
category_summary["share_receita"] = category_summary["receita"] / category_summary["receita"].sum()
category_summary["preco_medio_item"] = category_summary["receita"] / category_summary["itens"]
category_summary["ticket_medio"] = category_summary["receita"] / category_summary["pedidos"]
category_summary = category_summary.sort_values("receita", ascending=False)

display(category_summary)

fig, ax = plt.subplots(figsize=(10, 5))
ordered = category_summary.sort_values("preco_medio_item")
ax.barh(ordered["categoria_material_limpa"], ordered["preco_medio_item"])
ax.set_title("Preço médio por item — categoria")
ax.set_xlabel("Receita aprovada / itens")
fig.tight_layout()
plt.show()

In [ ]:
monthly = (
    daily.groupby("mes", as_index=False)
    .agg(
        receita=("receita_aprovada", "sum"),
        pedidos=("nr_pedidos", "sum"),
        itens=("qt_material", "sum"),
        dias=("data", "count"),
    )
)
monthly["preco_medio_item"] = monthly["receita"] / monthly["itens"]
monthly["demanda_media_diaria"] = monthly["itens"] / monthly["dias"]
display(monthly)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["mes"], monthly["preco_medio_item"], marker="o")
ax.set_title("Evolução mensal do preço médio por item")
ax.set_ylabel("Preço médio")
ax.set_xlabel("Mês")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

### 4.2 Potencial de vendas por dia da semana

Usamos **média e mediana**. A mediana é importante porque campanhas e datas especiais geram caudas longas. O `indice_potencial_itens` compara a mediana de cada dia com a mediana diária global.

In [ ]:
weekday = weekday_summary(daily)
display(weekday)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(weekday["dia_semana"], weekday["itens_mediana"])
ax.axhline(daily["qt_material"].median(), linestyle="--", label="Mediana global")
ax.set_title("Potencial de demanda por dia da semana")
ax.set_ylabel("Mediana de itens vendidos por dia")
ax.tick_params(axis="x", rotation=30)
ax.legend()
fig.tight_layout()
plt.show()

### 4.3 Flutuação ao longo do dia

Para não confundir volume do dia com perfil horário, calculamos também a **participação de cada hora no total diário**.

In [ ]:
hourly = hourly_summary(df)
display(hourly)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly["hora"], hourly["share_itens_media"] * 100, marker="o")
ax.set_title("Participação média da hora na demanda diária")
ax.set_xlabel("Hora")
ax.set_ylabel("% médio dos itens do dia")
ax.set_xticks(range(24))
fig.tight_layout()
plt.show()

### 4.4 Heatmap dia da semana × hora

Essa visão ajuda a identificar se o comportamento intradiário muda conforme o dia da semana.

In [ ]:
hour_dow = (
    df.groupby(["dia_semana_num", "dia_semana", "hora"], as_index=False)
    .agg(itens=("qt_material", "sum"))
)

totals = hour_dow.groupby(["dia_semana_num", "dia_semana"])["itens"].transform("sum")
hour_dow["share"] = hour_dow["itens"] / totals

heat = (
    hour_dow.pivot(index="dia_semana", columns="hora", values="share")
    .reindex(["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"])
)

fig, ax = plt.subplots(figsize=(14, 5))
image = ax.imshow(heat.to_numpy() * 100, aspect="auto")
ax.set_title("Distribuição da demanda por hora e dia da semana")
ax.set_xlabel("Hora")
ax.set_ylabel("Dia da semana")
ax.set_xticks(range(24))
ax.set_xticklabels(range(24))
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
cbar = fig.colorbar(image, ax=ax)
cbar.set_label("% da demanda do respectivo dia da semana")
fig.tight_layout()
plt.show()

## 5. Outros insights de negócio

### 5.1 Mix de canal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(channel_summary["DES_CANAL_VENDA_FINAL_AGRUP"], channel_summary["share_receita"] * 100)
axes[0].set_title("Participação de receita por canal")
axes[0].set_ylabel("% da receita")

channel_items = (
    df.groupby("DES_CANAL_VENDA_FINAL_AGRUP")["qt_material"].sum()
    / df["qt_material"].sum()
)
axes[1].bar(channel_items.index, channel_items.values * 100)
axes[1].set_title("Participação de itens por canal")
axes[1].set_ylabel("% dos itens")

fig.tight_layout()
plt.show()

### 5.2 Mix de categoria — somente registros com categoria válida

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ordered = category_summary.sort_values("share_receita")
ax.barh(ordered["categoria_material_limpa"], ordered["share_receita"] * 100)
ax.set_title("Participação de receita por categoria")
ax.set_xlabel("% da receita atribuível às categorias válidas")
fig.tight_layout()
plt.show()

### 5.3 Tendência mensal e possíveis eventos

In [ ]:
monthly_daily = (
    daily.groupby("mes", as_index=False)
    .agg(
        receita_media_diaria=("receita_aprovada", "mean"),
        itens_media_diaria=("qt_material", "mean"),
        pedidos_media_diaria=("nr_pedidos", "mean"),
    )
)
display(monthly_daily)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_daily["mes"], monthly_daily["itens_media_diaria"], marker="o")
ax.set_title("Demanda média diária por mês")
ax.set_ylabel("Itens/dia")
ax.set_xlabel("Mês")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

## 6. Relações entre métricas

Correlação serve para entendimento do comportamento, **não para definir features do modelo sem considerar disponibilidade temporal**.

In [ ]:
corr = daily[
    [
        "receita_aprovada",
        "nr_pedidos",
        "qt_material",
        "vlr_venda_desconto",
        "preco_medio_item",
        "ticket_medio",
        "itens_por_pedido",
    ]
].corr(method="spearman")

display(corr)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Correlação Spearman — métricas diárias")

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()

## 7. Model Readiness — decisões para a próxima etapa

### Target
**Target operacional proposto:** `qt_material` agregado por dia, por representar diretamente a demanda em unidades.

`receita_aprovada` será mantida como KPI financeiro e poderá ser modelada em paralelo como teste de sensibilidade.

### Leakage
Para prever o dia *t*, não podemos usar métricas realizadas do próprio dia *t* (`receita_aprovada`, `nr_pedidos`, `qt_material`, `vlr_venda_desconto`, mix realizado por canal/categoria). Essas informações só existem após a venda.

### Features candidatas
- calendário: dia da semana, mês, fim de semana;
- lags da demanda: t-1, t-7, t-14, t-28;
- rolling mean/std calculados **somente com passado**;
- lags de preço médio, receita e pedidos;
- tendência temporal;
- posteriormente, datas comerciais/feriados se forem adicionadas como fonte externa conhecida antes da previsão.

### Validação
Não usar `train_test_split` aleatório. A validação deve ser temporal, preferencialmente com **rolling/expanding backtest**.

### Baselines
Comparar qualquer ML contra pelo menos:
- naive `t-1`;
- naive sazonal `t-7`;
- média móvel histórica.

### Métricas
Para demanda diária, priorizar **MAE, WAPE, Bias e Skill vs baseline**. MAPE deve ser usado com cautela em séries com valores próximos de zero.

## 8. Resumo executivo da EDA

In [ ]:
top_day = daily.loc[daily["qt_material"].idxmax()]
bottom_day = daily.loc[daily["qt_material"].idxmin()]
top_weekday = weekday.loc[weekday["itens_mediana"].idxmax()]
bottom_weekday = weekday.loc[weekday["itens_mediana"].idxmin()]
peak_hour = hourly.loc[hourly["share_itens_media"].idxmax()]

executive_summary = pd.DataFrame([
    {
        "tema": "Data Quality",
        "insight": f"{df['categoria_invalida'].mean():.2%} das linhas possuem categoria inválida.",
        "implicacao": "Não usar categoria bruta sem tratamento; preservar linhas nos totais.",
    },
    {
        "tema": "Receita negativa",
        "insight": f"{df['receita_negativa'].mean():.2%} das linhas têm receita negativa.",
        "implicacao": "Validar significado com negócio; não remover automaticamente.",
    },
    {
        "tema": "Maior dia de demanda",
        "insight": f"{top_day['data'].date()} — {int(top_day['qt_material']):,} itens.",
        "implicacao": "Picos devem ser entendidos como eventos, não descartados por outlier.",
    },
    {
        "tema": "Dia da semana",
        "insight": f"Maior mediana: {top_weekday['dia_semana']}; menor: {bottom_weekday['dia_semana']}.",
        "implicacao": "Dia da semana é feature de calendário relevante.",
    },
    {
        "tema": "Intradiário",
        "insight": f"Hora de maior participação média: {int(peak_hour['hora'])}h.",
        "implicacao": "Existe padrão horário claro; útil para entendimento operacional.",
    },
])

display(executive_summary)